Setup and raw data

In [1]:
import os, sys, shutil
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = "google.colab" in sys.modules
BASE = Path("smart-pharmacy-big-data")
for d in ["data/raw", "data/processed", "data/simulated"]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

RAW = BASE / "data/raw/salesdaily.csv"
print("Running in Colab:", IN_COLAB)

Running in Colab: True


In [2]:
if not RAW.exists():
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()            # select salesdaily.csv
        for name in uploaded:
            shutil.move(name, BASE / "data/raw" / name)
    else:
        raise FileNotFoundError(f"Copy salesdaily.csv into {RAW.parent} and re-run this cell.")
print("Raw file:", RAW, "-", RAW.stat().st_size, "bytes")

Saving salesdaily.csv to salesdaily.csv
Raw file: smart-pharmacy-big-data/data/raw/salesdaily.csv - 117854 bytes


In [4]:
raw = pd.read_csv(RAW)
print("Shape:", raw.shape)
print("\nColumns:", list(raw.columns))
print("\nTypes:\n", raw.dtypes)
display(raw.head())
print("\nMissing values per column:\n", raw.isna().sum())
print("\nFully duplicated rows:", raw.duplicated().sum())
display(raw.describe())

Shape: (2106, 13)

Columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']

Types:
 datum            object
M01AB           float64
M01AE           float64
N02BA           float64
N02BE           float64
N05B            float64
N05C            float64
R03             float64
R06             float64
Year              int64
Month             int64
Hour              int64
Weekday Name     object
dtype: object


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday



Missing values per column:
 datum           0
M01AB           0
M01AE           0
N02BA           0
N02BE           0
N05B            0
N05C            0
R03             0
R06             0
Year            0
Month           0
Hour            0
Weekday Name    0
dtype: int64

Fully duplicated rows: 0


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour
count,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000,2106.000000
mean,5.033683,3.895830,3.880441,29.917095,8.853627,0.593522,5.512262,2.900198,2016.401235,6.344255,275.945869
std,2.737579,2.133337,2.384010,15.590966,5.605605,1.092988,6.428736,2.415816,1.665060,3.386954,1.970547
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2014.000000,1.000000,190.000000
25%,3.000000,2.340000,2.000000,19.000000,5.000000,0.000000,1.000000,1.000000,2015.000000,3.000000,276.000000
50%,4.990000,3.670000,3.500000,26.900000,8.000000,0.000000,4.000000,2.000000,2016.000000,6.000000,276.000000
75%,6.670000,5.138000,5.200000,38.300000,12.000000,1.000000,8.000000,4.000000,2018.000000,9.000000,276.000000
max,17.340000,14.463000,16.000000,161.000000,54.833333,9.000000,45.000000,15.000000,2019.000000,12.000000,276.000000


In [5]:
EXPECTED_ATC = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
missing_cols = [c for c in EXPECTED_ATC + ["datum"] if c not in raw.columns]
if missing_cols:
    raise ValueError(f"Unexpected format, columns not found: {missing_cols}. Check raw.columns above.")
ATC_COLS = EXPECTED_ATC
print("date column 'datum' + 8 ATC columns found")

date column 'datum' + 8 ATC columns found


 Cleaning

In [6]:
df = raw.copy()
df.columns = [c.strip() for c in df.columns]
log = {}

# Rule 1: parse dates (fallback to day-first if most values fail)
parsed = pd.to_datetime(df["datum"], errors="coerce")
if parsed.isna().mean() > 0.5:
    parsed = pd.to_datetime(df["datum"], errors="coerce", dayfirst=True)
df["datum"] = parsed
log["invalid_dates_dropped"] = int(df["datum"].isna().sum())
df = df.dropna(subset=["datum"])

# Rule 2: one row per day -> drop duplicated dates
log["duplicate_dates_dropped"] = int(df.duplicated(subset="datum").sum())
df = df.drop_duplicates(subset="datum").sort_values("datum").reset_index(drop=True)

# Rule 3: quantities must be numeric
for c in ATC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Rule 4: missing quantities -> 0 (no recorded sale that day)
log["missing_quantities_filled_with_0"] = int(df[ATC_COLS].isna().sum().sum())
df[ATC_COLS] = df[ATC_COLS].fillna(0)

# Rule 5: negative quantities are invalid -> set to 0
log["negative_quantities_set_to_0"] = int((df[ATC_COLS] < 0).sum().sum())
df[ATC_COLS] = df[ATC_COLS].clip(lower=0)

# Check: are there gaps in the daily calendar?
full_range = pd.date_range(df["datum"].min(), df["datum"].max(), freq="D")
log["calendar_days_missing"] = int(len(full_range) - len(df))
log["rows_after_cleaning"] = int(len(df))
log["date_min"] = str(df["datum"].min().date())
log["date_max"] = str(df["datum"].max().date())

wide = df[["datum"] + ATC_COLS].copy()
print(pd.Series(log))
display(wide.head())

invalid_dates_dropped                        0
duplicate_dates_dropped                      0
missing_quantities_filled_with_0             0
negative_quantities_set_to_0                 0
calendar_days_missing                        0
rows_after_cleaning                       2106
date_min                            2014-01-02
date_max                            2019-10-08
dtype: object


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0


In [7]:
medicines = pd.DataFrame({
    "medicine_id": [f"M{i:03d}" for i in range(1, 9)],
    "atc_code": ATC_COLS,
    "category_name": [
        "Anti-inflammatory/antirheumatic: acetic acid derivatives",
        "Anti-inflammatory/antirheumatic: propionic acid derivatives",
        "Analgesics: salicylic acid and derivatives",
        "Analgesics: pyrazolones and anilides",
        "Anxiolytics",
        "Hypnotics and sedatives",
        "Drugs for obstructive airway diseases",
        "Antihistamines for systemic use",
    ],
})
display(medicines)

sales_long = (
    wide.melt(id_vars="datum", value_vars=ATC_COLS, var_name="atc_code", value_name="quantity")
        .rename(columns={"datum": "sale_date"})
        .merge(medicines[["medicine_id", "atc_code"]], on="atc_code", how="left")
        [["sale_date", "medicine_id", "atc_code", "quantity"]]
        .sort_values(["sale_date", "medicine_id"])
        .reset_index(drop=True)
)
sales_long["quantity"] = sales_long["quantity"].round(2)
sales_long["sale_date"] = sales_long["sale_date"].dt.strftime("%Y-%m-%d")
print(sales_long.shape)
display(sales_long.head(10))

,medicine_id,atc_code,category_name
0,M001,M01AB,Anti-inflammatory/antirheumatic: acetic acid d...
1,M002,M01AE,Anti-inflammatory/antirheumatic: propionic aci...
2,M003,N02BA,Analgesics: salicylic acid and derivatives
3,M004,N02BE,Analgesics: pyrazolones and anilides
4,M005,N05B,Anxiolytics
5,M006,N05C,Hypnotics and sedatives
6,M007,R03,Drugs for obstructive airway diseases
7,M008,R06,Antihistamines for systemic use


(16848, 4)


,sale_date,medicine_id,atc_code,quantity
0,2014-01-02,M001,M01AB,0.00
1,2014-01-02,M002,M01AE,3.67
2,2014-01-02,M003,N02BA,3.40
3,2014-01-02,M004,N02BE,32.40
4,2014-01-02,M005,N05B,7.00
5,2014-01-02,M006,N05C,0.00
6,2014-01-02,M007,R03,0.00
7,2014-01-02,M008,R06,2.00
8,2014-01-03,M001,M01AB,8.00
9,2014-01-03,M002,M01AE,4.00


In [8]:
# Save the processed files (the raw file stays untouched in data/raw/)
sales_long.to_csv(BASE / "data/processed/pharma_sales_clean.csv", index=False)
wide.assign(datum=wide["datum"].dt.strftime("%Y-%m-%d")).to_csv(
    BASE / "data/processed/pharma_sales_clean_wide.csv", index=False)
medicines.to_csv(BASE / "data/simulated/medicines.csv", index=False)
print("Saved processed files.")

Saved processed files.


## Step 4 — Simulated pharmacies and initial stocks
These are **fictional** (the dataset comes from one real pharmacy and has no stock data). Seeded, so everyone gets the same files.
The demo case from the brief is forced: **P001 / M001 → stock 100, threshold 30**.

In [9]:
pharmacies = pd.DataFrame({
    "pharmacy_id": [f"P{i:03d}" for i in range(1, 6)],
    "name": ["Pharmacie Centrale", "Pharmacie El Amel", "Pharmacie Ennour", "Pharmacie La Marsa", "Pharmacie Ibn Sina"],
    "city": ["Tunis", "Sfax", "Sousse", "Bizerte", "Nabeul"],
})

rng = np.random.default_rng(42)
rows = []
now = pd.Timestamp.now().strftime("%Y-%m-%dT%H:%M:%S")
for p in pharmacies["pharmacy_id"]:
    for m in medicines["medicine_id"]:
        rows.append({
            "pharmacy_id": p,
            "medicine_id": m,
            "quantity": int(rng.integers(150, 501)),
            "min_threshold": int(rng.choice([30, 40, 50])),
            "updated_at": now,
        })
initial_stock = pd.DataFrame(rows)

# Demo scenario from the brief
mask = (initial_stock["pharmacy_id"] == "P001") & (initial_stock["medicine_id"] == "M001")
initial_stock.loc[mask, ["quantity", "min_threshold"]] = [100, 30]

pharmacies.to_csv(BASE / "data/simulated/pharmacies.csv", index=False)
initial_stock.to_csv(BASE / "data/simulated/initial_stock.csv", index=False)
display(pharmacies)
display(initial_stock.head(10))
print("Demo row:\n", initial_stock[mask])

,pharmacy_id,name,city
0,P001,Pharmacie Centrale,Tunis
1,P002,Pharmacie El Amel,Sfax
2,P003,Pharmacie Ennour,Sousse
3,P004,Pharmacie La Marsa,Bizerte
4,P005,Pharmacie Ibn Sina,Nabeul


,pharmacy_id,medicine_id,quantity,min_threshold,updated_at
0,P001,M001,100,30,2026-11-07T14:52:01
1,P001,M002,379,40,2026-11-07T14:52:01
2,P001,M003,301,50,2026-11-07T14:52:01
3,P001,M004,180,50,2026-11-07T14:52:01
4,P001,M005,220,30,2026-11-07T14:52:01
5,P001,M006,334,50,2026-11-07T14:52:01
6,P001,M007,408,50,2026-11-07T14:52:01
7,P001,M008,401,50,2026-11-07T14:52:01
8,P002,M001,330,30,2026-11-07T14:52:01
9,P002,M002,444,40,2026-11-07T14:52:01


Demo row:
   pharmacy_id medicine_id  quantity  min_threshold           updated_at
0        P001        M001       100             30  2026-11-07T14:52:01


## Step 5 — Validate the shared contract

In [10]:
checks = {
    "stock key (pharmacy_id, medicine_id) is unique":
        initial_stock.duplicated(["pharmacy_id", "medicine_id"]).sum() == 0,
    "pharmacy_ids identical in pharmacies and stock":
        set(pharmacies.pharmacy_id) == set(initial_stock.pharmacy_id),
    "medicine_ids identical in medicines, stock and sales":
        set(medicines.medicine_id) == set(initial_stock.medicine_id) == set(sales_long.medicine_id),
    "stock quantities are positive integers":
        (initial_stock["quantity"] > 0).all(),
    "no negative sales quantities":
        (sales_long["quantity"] >= 0).all(),
    "no missing values in processed sales":
        sales_long.isna().sum().sum() == 0,
    "demo scenario P001/M001 = 100 with threshold 30":
        bool(initial_stock[mask][["quantity", "min_threshold"]].iloc[0].tolist() == [100, 30]),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values()), "Fix the failing checks before sharing the files."

PASS - stock key (pharmacy_id, medicine_id) is unique
PASS - pharmacy_ids identical in pharmacies and stock
PASS - medicine_ids identical in medicines, stock and sales
PASS - stock quantities are positive integers
PASS - no negative sales quantities
PASS - no missing values in processed sales
PASS - demo scenario P001/M001 = 100 with threshold 30


## Step 6 — Make it big: multi-pharmacy transaction history (simulated)
The real daily file is tiny (about 2,000 rows). To give Spark something worth distributing, we build a **transaction-level sales history** in the *same format as the Kafka events*:
`sale_id, pharmacy_id, medicine_id, quantity, timestamp`.

How it is built:
1. Take the **real hourly** sales pattern (`saleshourly.csv`): which categories sell, at which hours, in which seasons.
2. For each simulated pharmacy, draw units sold per hour with a Poisson distribution around that real pattern (each pharmacy has its own activity factor).
3. Split the units into individual sale lines (1 to 3 units each) with random minutes and seconds.

In [11]:
HOURLY = BASE / "data/raw/saleshourly.csv"
if not HOURLY.exists():
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded:
            shutil.move(name, BASE / "data/raw" / name)
    else:
        raise FileNotFoundError(f"Copy saleshourly.csv into {HOURLY.parent} and re-run this cell.")

hourly = pd.read_csv(HOURLY)
print("Hourly shape:", hourly.shape)
display(hourly.head())
missing_cols = [c for c in ["datum"] + ATC_COLS if c not in hourly.columns]
if missing_cols:
    raise ValueError(f"Unexpected format, columns not found: {missing_cols}")

Saving saleshourly.csv to saleshourly.csv
Hourly shape: (50532, 13)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014 8:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday
1,1/2/2014 9:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday
2,1/2/2014 10:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday
3,1/2/2014 11:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday
4,1/2/2014 12:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday


In [12]:
# Same cleaning rules as the daily file
hr = hourly[["datum"] + ATC_COLS].copy()
ts = pd.to_datetime(hr["datum"], errors="coerce")
if ts.isna().mean() > 0.5:
    ts = pd.to_datetime(hr["datum"], errors="coerce", dayfirst=True)
hr["ts_hour"] = ts
hr_invalid = int(hr["ts_hour"].isna().sum())
hr = hr.dropna(subset=["ts_hour"])
hr["ts_hour"] = hr["ts_hour"].dt.floor("60min")
hr_dups = int(hr.duplicated(subset="ts_hour").sum())
hr = hr.drop_duplicates(subset="ts_hour")
for c in ATC_COLS:
    hr[c] = pd.to_numeric(hr[c], errors="coerce")
hr_missing = int(hr[ATC_COLS].isna().sum().sum())
hr_negative = int((hr[ATC_COLS] < 0).sum().sum())
hr[ATC_COLS] = hr[ATC_COLS].fillna(0).clip(lower=0)

base_units = float(hr[ATC_COLS].sum().sum())
print("Hourly rows:", len(hr), "| from", hr["ts_hour"].min(), "to", hr["ts_hour"].max())
print("Invalid dates:", hr_invalid, "| duplicate hours:", hr_dups, "| missing:", hr_missing, "| negative:", hr_negative)
print("Total units in the real hourly file:", round(base_units))

Hourly rows: 50532 | from 2014-01-02 08:00:00 to 2019-10-08 19:00:00
Invalid dates: 0 | duplicate hours: 0 | missing: 0 | negative: 0
Total units in the real hourly file: 127596


In [13]:
SEED = 2026
TARGET_UNITS = 4_000_000

rng = np.random.default_rng(SEED)
ph_ids = pharmacies["pharmacy_id"].tolist()
ph_factor = dict(zip(ph_ids, rng.uniform(0.6, 1.5, len(ph_ids))))   # each pharmacy has its own activity level
scale = TARGET_UNITS / (base_units * sum(ph_factor.values()))
print("Activity factors:", {k: round(v, 2) for k, v in ph_factor.items()}, "| scale:", round(scale, 2))

long_h = (hr.melt(id_vars="ts_hour", value_vars=ATC_COLS, var_name="atc_code", value_name="rate")
            .merge(medicines[["medicine_id", "atc_code"]], on="atc_code"))
long_h = long_h[long_h["rate"] > 0].reset_index(drop=True)

frames = []
for p in ph_ids:
    units = rng.poisson(long_h["rate"].to_numpy() * ph_factor[p] * scale)
    f = long_h[["ts_hour", "medicine_id"]].copy()
    f["pharmacy_id"] = p
    f["units"] = units
    frames.append(f[f["units"] > 0])
cells_df = pd.concat(frames, ignore_index=True)
print("Hour/pharmacy/medicine cells with sales:", len(cells_df), "| units:", int(cells_df["units"].sum()))

Activity factors: {'P001': np.float64(0.76), 'P002': np.float64(1.18), 'P003': np.float64(1.02), 'P004': np.float64(0.93), 'P005': np.float64(0.92)} | scale: 6.52
Hour/pharmacy/medicine cells with sales: 324837 | units: 3999294


In [14]:
# Split units into individual sale lines (1 to 3 units each), with random minutes/seconds
BASKET = 3
cells_df["n_lines"] = np.ceil(cells_df["units"] / BASKET).astype(int)
tx = cells_df.loc[cells_df.index.repeat(cells_df["n_lines"])].copy()
tx["k"] = tx.groupby(level=0).cumcount()
tx["quantity"] = (tx["units"] // tx["n_lines"] + (tx["k"] < tx["units"] % tx["n_lines"]).astype(int)).astype(int)
tx["timestamp"] = tx["ts_hour"] + pd.to_timedelta(rng.integers(0, 3600, len(tx)), unit="s")
tx = tx.sort_values(["timestamp", "pharmacy_id", "medicine_id"]).reset_index(drop=True)
tx["sale_id"] = ["H" + str(i).zfill(8) for i in range(1, len(tx) + 1)]
tx["timestamp"] = tx["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")
history = tx[["sale_id", "pharmacy_id", "medicine_id", "quantity", "timestamp"]]

out = BASE / "data/processed/sales_history_transactions.csv"
history.to_csv(out, index=False)
print(f"{len(history):,} sale lines written to {out} ({out.stat().st_size/1e6:.0f} MB)")
display(history.head())

1,443,519 sale lines written to smart-pharmacy-big-data/data/processed/sales_history_transactions.csv (61 MB)


,sale_id,pharmacy_id,medicine_id,quantity,timestamp
0,H00000001,P005,M004,3,2014-01-02T08:01:11
1,H00000002,P002,M004,3,2014-01-02T08:04:30
2,H00000003,P002,M003,2,2014-01-02T08:05:02
3,H00000004,P005,M008,3,2014-01-02T08:05:26
4,H00000005,P005,M004,3,2014-01-02T08:06:54


## Step 7 — Validate the history

In [15]:
hist_checks = {
    "sale_id is unique": history["sale_id"].is_unique,
    "quantity is an integer > 0": bool((history["quantity"] > 0).all()),
    "pharmacy_id and medicine_id exist in the reference files":
        set(history.pharmacy_id) <= set(pharmacies.pharmacy_id) and set(history.medicine_id) <= set(medicines.medicine_id),
    "no missing values": int(history.isna().sum().sum()) == 0,
    "total units preserved when splitting": int(history["quantity"].sum()) == int(cells_df["units"].sum()),
}
for name, ok in hist_checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(hist_checks.values())

hist_rows = len(history)
hist_units = int(history["quantity"].sum())
hist_min, hist_max = history["timestamp"].min(), history["timestamp"].max()
print("\nRows:", f"{hist_rows:,}", "| units:", f"{hist_units:,}", "| period:", hist_min, "->", hist_max)
display(history.groupby("pharmacy_id")["quantity"].agg(["count", "sum"]))
display(history.groupby("medicine_id")["quantity"].agg(["count", "sum"]))

PASS - sale_id is unique
PASS - quantity is an integer > 0
PASS - pharmacy_id and medicine_id exist in the reference files
PASS - no missing values
PASS - total units preserved when splitting

Rows: 1,443,519 | units: 3,999,294 | period: 2014-01-02T08:01:11 -> 2019-10-08T19:59:40


,count,sum
pharmacy_id,,
P001,232454,631101
P002,347769,976878
P003,305189,849225
P004,280765,776182
P005,277342,765908


,count,sum
medicine_id,,
M001,125485,332106
M002,101373,256880
M003,97966,256828
M004,692250,1974819
M005,212446,584414
M006,14388,38846
M007,126912,363536
M008,72699,191865


## Step 8 — Generate `data/README_DATA.md`

In [16]:
lines = [
    "# Data README - Smart Pharmacy",
    "",
    "## Source (real data)",
    "- Dataset: *Pharma Sales Data* (Milan Zdravkovic, Kaggle), file `salesdaily.csv`.",
    "- One real pharmacy, 2014-2019, quantities aggregated into 8 ATC categories.",
    "- Raw file kept intact in `data/raw/` (not committed to Git).",
    "",
    "## Cleaning rules (data/processed/)",
    f"- Rows after cleaning: {log['rows_after_cleaning']} ({log['date_min']} to {log['date_max']}).",
    f"- Invalid dates dropped: {log['invalid_dates_dropped']}; duplicated dates dropped: {log['duplicate_dates_dropped']}.",
    f"- Missing quantities filled with 0: {log['missing_quantities_filled_with_0']}.",
    f"- Negative quantities set to 0: {log['negative_quantities_set_to_0']}.",
    f"- Calendar days absent from the file: {log['calendar_days_missing']}.",
    "- `pharma_sales_clean.csv`: long format `sale_date, medicine_id, atc_code, quantity` (quantities are daily totals and may be decimal).",
    "- `pharma_sales_clean_wide.csv`: same data, one column per ATC code.",
    "- Zero-quantity rows are kept; filter them in Spark (Filtering pattern).",
    "",
    "## Big transaction history (SIMULATED from real hourly data)",
    "- Real base: `saleshourly.csv` (hourly sales per ATC category, same cleaning rules as the daily file).",
    f"- Hourly rows used: {len(hr)}; invalid dates: {hr_invalid}; duplicate hours: {hr_dups}; missing filled with 0: {hr_missing}; negatives set to 0: {hr_negative}.",
    "- `sales_history_transactions.csv`: one row per sale, columns `sale_id, pharmacy_id, medicine_id, quantity, timestamp` (same format as the Kafka events).",
    f"- Size: {hist_rows:,} sale lines, {hist_units:,} units, from {hist_min} to {hist_max}.",
    "- Method: units per hour, pharmacy and medicine are drawn from a Poisson distribution around the real hourly pattern, scaled by a per-pharmacy activity factor (seed 2026), then split into sale lines of 1 to 3 units with random minutes and seconds.",
    "- History `sale_id`s start with `H` so they never collide with live Kafka ids (`S...`).",
    "- This history is simulated; only the seasonal and daily pattern comes from real data.",
    "",
    "## Simulated data (NOT real)",
    "- `pharmacies.csv`: 5 fictional pharmacies (P001-P005).",
    "- `initial_stock.csv`: random initial stocks and minimum thresholds (seed 42), one row per pharmacy/medicine.",
    "- Demo case: P001 / M001 has stock 100 and threshold 30.",
    "- `medicines.csv`: `medicine_id` (M001-M008) assigned by the project, one per ATC category.",
    "- Real-time sale events (Kafka) are also simulated.",
    "",
    "## Shared contract",
    "- `pharmacy_id` (P001...) and `medicine_id` (M001...) are identical in every file and service.",
    "- Event format: `{\"sale_id\": \"S000001\", \"pharmacy_id\": \"P001\", \"medicine_id\": \"M001\", \"quantity\": 2, \"timestamp\": \"2026-11-06T10:30:00\"}`",
]
(BASE / "data/README_DATA.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

# Data README - Smart Pharmacy

## Source (real data)
- Dataset: *Pharma Sales Data* (Milan Zdravkovic, Kaggle), file `salesdaily.csv`.
- One real pharmacy, 2014-2019, quantities aggregated into 8 ATC categories.
- Raw file kept intact in `data/raw/` (not committed to Git).

## Cleaning rules (data/processed/)
- Rows after cleaning: 2106 (2014-01-02 to 2019-10-08).
- Invalid dates dropped: 0; duplicated dates dropped: 0.
- Missing quantities filled with 0: 0.
- Negative quantities set to 0: 0.
- Calendar days absent from the file: 0.
- `pharma_sales_clean.csv`: long format `sale_date, medicine_id, atc_code, quantity` (quantities are daily totals and may be decimal).
- `pharma_sales_clean_wide.csv`: same data, one column per ATC code.
- Zero-quantity rows are kept; filter them in Spark (Filtering pattern).

## Big transaction history (SIMULATED from real hourly data)
- Real base: `saleshourly.csv` (hourly sales per ATC category, same cleaning rules as the daily file).
- Hourly rows used

## Step 9 — Package the outputs

In [17]:
tmp = Path("export_membre1")
if tmp.exists():
    shutil.rmtree(tmp)
shutil.copytree(BASE / "data", tmp / "data", ignore=shutil.ignore_patterns("raw"))
shutil.make_archive("membre1_data_outputs", "zip", tmp)
shutil.rmtree(tmp)
print("Created membre1_data_outputs.zip")

if IN_COLAB:
    from google.colab import files
    files.download("membre1_data_outputs.zip")

Created membre1_data_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>